# PyMIF notebook 05 - add image groups and write image regions

Image groups are useful for derived intensity-like data, for example:

- denoised images,
- deconvolved images,
- probability maps,
- illumination-corrected images.

They live at the root level next to the raw image, not under `/labels`.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

In [ ]:
def make_small_raw_manager(seed=0, num_levels=2):
    """Create a tiny TCZYX intensity dataset for examples."""
    rng = np.random.default_rng(seed)
    raw = rng.integers(0, 1200, size=(2, 3, 4, 64, 64), dtype=np.uint16)
    metadata = {
        "name": "small_raw",
        "axes": "tczyx",
        "size": [raw.shape],
        "chunksize": [(1, 1, 2, 32, 32)],
        "scales": [(2.0, 0.30, 0.30)],   # z, y, x spacing because axes contains zyx
        "units": ("micrometer", "micrometer", "micrometer"),
        "time_increment": 60.0,
        "time_increment_unit": "second",
        "channel_names": ["DAPI", "GFP", "RFP"],
        "channel_colors": ["0000FF", "00FF00", "FF0000"],
        "dtype": "uint16",
        "data_type": "intensity",
    }
    manager = mm.ArrayManager(raw, metadata, chunks=metadata["chunksize"][0])
    if num_levels > 1:
        manager.build_pyramid(num_levels=num_levels, downscale_factor=(1, 2, 2))
    return manager

## Create a root image Zarr

In [ ]:
raw_manager = make_small_raw_manager(seed=11, num_levels=2)
path = clean_path(OUTPUT_DIR / "groups_example.zarr")
raw_manager.to_zarr(path, ngff_version="0.5", zarr_format=3)
z = mm.ZarrManager(path, mode="r+")
summarize_manager(z, "root image before groups")

## Add an empty image group

For an image group, keep `data_type="intensity"` and usually keep the channel axis.

In [ ]:
prob_md = dict(z.metadata)
prob_md["name"] = "probability"
prob_md["dtype"] = "float32"
prob_md["data_type"] = "intensity"
prob_md["channel_names"] = ["nucleus_probability", "cell_probability", "background_probability"]
prob_md["channel_colors"] = ["00FFFF", "FF00FF", "FFFFFF"]

probability = z.create_empty_group("probability", prob_md, is_label=False)
print(probability)
print("Current image groups:", list(z.groups.keys()))

## Write the whole group in one call

When a single level-0 array is supplied, `write_image_region` fills the full pyramid by downsampling to the other levels.

In [ ]:
shape = prob_md["size"][0]
prob_data = np.zeros(shape, dtype=np.float32)
prob_data[:, 0, :, 10:45, 10:45] = 0.9
prob_data[:, 1, :, 35:60, 20:55] = 0.7
prob_data[:, 2, :, :, :] = 0.1

z.write_image_region(
    prob_data,
    group="probability",
    level=0,
    downscale_factor=(1, 2, 2),
)
z.read()
print("After writing probability group:", list(z.groups.keys()))
print(z.groups["probability"])

## Write only a smaller region

The passed data shape must exactly match the selected region. Here we update one time point, one channel, all Z, and a small YX box.

In [ ]:
patch = np.ones((1, 1, shape[2], 16, 16), dtype=np.float32)
z = mm.ZarrManager(path, mode="r+")
z.write_image_region(
    patch,
    t=slice(0, 1),
    c=slice(0, 1),
    z=slice(None),
    y=slice(8, 24),
    x=slice(8, 24),
    group="probability",
    level=0,
    downscale_factor=(1, 2, 2),
)
z.read()
updated = z.groups["probability"].data[0][0, 0, 0, 8:24, 8:24].compute()
print("Updated patch min/max:", updated.min(), updated.max())

## Copy the complete store, including groups

`ZarrManager.to_zarr()` writes raw data, image groups, and label groups into a new complete store.

In [ ]:
copy_path = clean_path(OUTPUT_DIR / "groups_example_copy.zarr")
z.to_zarr(copy_path, include_groups=True, include_labels=True, ngff_version="0.5", zarr_format=3)
copy_z = mm.ZarrManager(copy_path, mode="r")
print("Copied groups:", list(copy_z.groups.keys()))